# Chapter 4 — Getting Indian Data into Python

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch04_getting_indian_data.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 4.1 Where official Indian data lives

## 4.2 Reading a CSV file

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from indiadata import bis

bis("WS_CBPOL", "M.IN")                      # downloads once, then uses the saved copy
path = Path("data/raw/bis_WS_CBPOL_M.IN.csv")
print(path.read_text().splitlines()[:3])     # the first three lines, as raw text

repo = pd.read_csv(path, parse_dates=["date"])
print(repo.dtypes)
repo.tail(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
recent = repo[repo["date"] >= "2000-01-01"]
ax.step(recent["date"], recent["value"], where="post", color="#1f4e79",
        linewidth=1.8)
ax.set_title("RBI policy rate, % per year (monthly)", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Source: Bank for International Settlements, central bank "
         "policy rates (WS_CBPOL), series M.IN; read from a saved CSV file.",
         fontsize=8, color="grey")
plt.show()

## 4.3 Reading an Excel file with a messy top

In [ ]:
from indiadata import mospi_iip

iip_annual = mospi_iip(frequency="annual", base_year="2011-12")
general = (iip_annual[iip_annual["type"] == "General"][["year", "index", "growth_rate"]]
           .sort_values("year").reset_index(drop=True))

xlsx = Path("data/raw/practice_iip_annual.xlsx")
with pd.ExcelWriter(xlsx) as writer:
    general.rename(columns={"year": "Year", "index": "Index",
                            "growth_rate": "Growth (%)"}).to_excel(
        writer, sheet_name="IIP", startrow=3, index=False)
    sheet = writer.sheets["IIP"]
    sheet["A1"] = "Index of Industrial Production: General Index (annual)"
    sheet["A2"] = "Base: 2011-12 = 100"
    sheet.cell(row=len(general) + 6, column=1,
               value="Note: practice file made for Chapter 4 from MoSPI API data.")
    pd.DataFrame({"Notes": ["Source: MoSPI eSankhyiki API, iip/getIIPAnnual"]}
                 ).to_excel(writer, sheet_name="Notes", index=False)

print(pd.ExcelFile(xlsx).sheet_names)        # which sheets are in the file?
pd.read_excel(xlsx, header=None).head(5)     # look at the top, with no header

In [ ]:
iip_xl = pd.read_excel(xlsx, sheet_name="IIP", skiprows=3, skipfooter=2)
print(iip_xl.dtypes)
iip_xl.tail(3)

## 4.4 What is an API?

In [ ]:
import requests

MOSPI = "https://api.mospi.gov.in/api/"
HEADERS = {"User-Agent": "python-for-indian-economics (teaching use)"}

params = {"base_year": 2012, "series": "Current", "state_code": 99,
          "sector_code": 3, "group_code": 0, "Format": "JSON",
          "limit": 50, "page": 1}
reply = requests.get(MOSPI + "cpi/getCPIIndex", params=params,
                     headers=HEADERS, timeout=60)
print("Status code: ", reply.status_code)
print("Content type:", reply.headers["Content-Type"])
print("Full address:", reply.url)

body = reply.json()                  # JSON text -> Python dicts and lists
print(body.keys())
print(body["meta_data"])
page1 = pd.DataFrame(body["data"])   # a list of dicts becomes a DataFrame
page1[["year", "month", "index", "inflation", "status"]].head(3)

## 4.5 Pagination: getting every page

In [ ]:
import datetime as dt
import time

raw_path = Path("data/raw/ch04_cpi_all_groups_raw.csv")
if raw_path.exists():
    allg = pd.read_csv(raw_path, dtype=str)
    print("Read the saved copy fetched on", allg["fetched_on"].iloc[0])
    total = int(allg["total_records"].iloc[0])
else:
    q = {**params, "limit": 200}             # a copy of params, limit replaced
    del q["group_code"]                      # all groups this time
    rows, page = [], 1
    while True:
        reply = requests.get(MOSPI + "cpi/getCPIIndex", params={**q, "page": page},
                             headers=HEADERS, timeout=90)
        reply.raise_for_status()             # stop with an error if not 2xx
        body = reply.json()
        rows.extend(body["data"])            # add every row of this page
        if page >= body["meta_data"]["totalPages"]:
            break
        page += 1
        time.sleep(0.5)                      # be polite: pause between pages
    total = body["meta_data"]["totalRecords"]
    allg = pd.DataFrame(rows).astype(str)
    allg["total_records"] = str(total)
    allg["fetched_on"] = dt.date.today().isoformat()
    allg.to_csv(raw_path, index=False)
    print(f"Downloaded {page} pages")

allg = allg.drop(columns=["total_records", "fetched_on"])
print(f"Rows collected: {len(allg)} | server's totalRecords: {total}")

In [ ]:
keys = ["year", "month", "group", "subgroup"]
print("Rows repeated on the key columns:", allg.duplicated(keys).sum())
print("Rows repeated in every column:   ", allg.duplicated().sum())
allg[allg.duplicated(keys, keep=False)].sort_values(keys)[
    keys + ["index", "inflation", "status"]].head(4)

In [ ]:
allg = allg.drop_duplicates()
allg["date"] = pd.to_datetime(allg["year"] + "-" + allg["month"], format="%Y-%B")
print("Distinct rows:", len(allg))

months = pd.date_range(allg["date"].min(), allg["date"].max(), freq="MS")
for (g, sg), d in allg.groupby(["group", "subgroup"]):
    missing = months.difference(d["date"])
    if len(missing):
        print(f"{sg:22s} missing: {list(missing.strftime('%Y-%m'))}")

## 4.6 Polite and robust downloading

In [ ]:
try:
    requests.get(MOSPI + "iip/getIIPAnnual", params={"base_year": "2011-12"},
                 headers=HEADERS, timeout=0.001)
except requests.exceptions.Timeout:
    print("The server did not answer in time. Try again later.")
except requests.exceptions.RequestException as err:
    print("Some other network problem:", type(err).__name__)

wrong = requests.get(MOSPI + "iip/getIIPMonthlyX",            # a typo
                     params={"base_year": "2022-23"}, headers=HEADERS,
                     timeout=60)
print(wrong.status_code, wrong.headers["Content-Type"], wrong.text[:40])

empty = requests.get(MOSPI + "iip/getIIPAnnual",
                     params={"base_year": "1999-00", "Format": "JSON"},
                     headers=HEADERS, timeout=60)
print(empty.status_code, empty.json())

## 4.7 Downloading a file, and codes with leading zeros

In [ ]:
from pypdf import PdfReader

PR_URL = ("https://mospi.gov.in/sites/default/files/press_release/"
          "cpi_pr_12feb15t.pdf")
pdf_path = Path("data/raw/cpi_pr_12feb15t.pdf")

if not pdf_path.exists():                    # download only once
    reply = requests.get(PR_URL, headers=HEADERS, timeout=90)
    reply.raise_for_status()
    if "pdf" not in reply.headers.get("Content-Type", ""):
        raise ValueError("Expected a PDF, got " + reply.headers["Content-Type"])
    pdf_path.write_bytes(reply.content)      # .content = raw bytes of the file

text = PdfReader(pdf_path).pages[3].extract_text()
lines = [ln for ln in text.splitlines() if ln[:2].isdigit()]
print(len(lines), "lines start with a code; the first is:")
print(lines[0])

records = []
for ln in lines:
    parts = ln.split()
    code, name, numbers = parts[0], " ".join(parts[1:-9]), parts[-9:]
    records.append({"code": code, "state": name,
                    "w_rural": numbers[0], "w_urban": numbers[3],
                    "w_combined": numbers[6]})
csv_path = Path("data/raw/cpi2012_state_weights.csv")
pd.DataFrame(records).to_csv(csv_path, index=False)

back = pd.read_csv(csv_path)                 # read it back, as anyone would
back[["code", "state"]].head(3)

In [ ]:
back = pd.read_csv(csv_path, dtype={"code": str})
print(back["code"].head(3).tolist())

num = back[back["code"] != "99"].copy()       # drop the All India row
print(num[["w_rural", "w_urban", "w_combined"]].sum().round(2))   # 100 each?

In [ ]:
top = num.sort_values("w_combined").tail(12)
fig, ax = plt.subplots(figsize=(9, 5))
y = range(len(top))
ax.barh([i + 0.2 for i in y], top["w_rural"], height=0.4, color="#9db7d5",
        label="Rural CPI")
ax.barh([i - 0.2 for i in y], top["w_urban"], height=0.4, color="#1f4e79",
        label="Urban CPI")
ax.set_yticks(list(y), top["state"])
ax.set_xlabel("Weight in all-India index, %")
ax.set_title("Weight of the 12 largest states in the CPI (base 2012)",
             loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Source: MoSPI, CPI press release of 12 February 2015, "
         "Annexure III; states ordered by combined weight.",
         fontsize=8, color="grey")
plt.show()

print(num.set_index("state").loc[["Maharashtra", "Uttar Pradesh", "Bihar"],
                                 ["w_rural", "w_urban"]])

## 4.8 Opening the box: how `indiadata` works

In [ ]:
import inspect
import indiadata.sources as src

print("\n".join(inspect.getsource(src.mospi).splitlines()[:31]))

## 4.9 Write your own fetcher

In [ ]:
def fetch_iip_annual(base_year, refresh=False):
    """Annual IIP (all categories) from MoSPI, cached in data/raw/."""
    path = Path(f"data/raw/my_iip_annual_{base_year}.csv")
    if path.exists() and not refresh:                      # 1. use the cache
        return pd.read_csv(path)

    rows, page = [], 1
    while True:                                            # 2. every page
        try:
            reply = requests.get(MOSPI + "iip/getIIPAnnual",
                                 params={"base_year": base_year, "Format": "JSON",
                                         "limit": 200, "page": page},
                                 headers=HEADERS, timeout=60)
            reply.raise_for_status()
        except requests.exceptions.RequestException as err:
            raise RuntimeError(f"Download failed on page {page}: {err}")
        if "json" not in reply.headers.get("Content-Type", ""):   # 3. JSON?
            raise ValueError("The server did not send JSON. Check the endpoint.")
        body = reply.json()
        rows.extend(body.get("data") or [])
        if page >= (body.get("meta_data") or {}).get("totalPages", 1):
            break
        page += 1
        time.sleep(0.5)

    if not rows:                                           # 4. not empty?
        raise ValueError(f"No data for base year {base_year}.")
    df = pd.DataFrame(rows)
    keys = ["year", "type", "category", "sub_category"]
    if df.duplicated(keys).any():                          # 5. no duplicates?
        raise ValueError("Duplicate rows: inspect them before dropping.")

    df["index"] = pd.to_numeric(df["index"], errors="coerce")
    df["growth_rate"] = pd.to_numeric(df["growth_rate"], errors="coerce")
    df["fetched_on"] = dt.date.today().isoformat()         # 6. record the date
    df["source"] = MOSPI + "iip/getIIPAnnual"
    df.to_csv(path, index=False)
    return df


mine = fetch_iip_annual("2011-12")
print(len(mine), "rows; fetched on", mine["fetched_on"].iloc[0])

In [ ]:
monthly = mospi_iip(frequency="monthly", base_year="2011-12")
gm = monthly[monthly["type"] == "General"].copy()
# fiscal year: April 2023 to March 2024 is "2023-24"
start = gm["date"].dt.year.where(gm["date"].dt.month >= 4, gm["date"].dt.year - 1)
gm["year"] = start.astype(str) + "-" + (start + 1).astype(str).str[-2:]
from_monthly = gm.groupby("year")["index"].agg(["mean", "count"])

check = (mine[mine["type"] == "General"].set_index("year")[["index"]]
         .join(from_monthly).rename(columns={"index": "annual (MoSPI)",
                                             "mean": "mean of months"}))
check["gap"] = (check["annual (MoSPI)"] - check["mean of months"]).round(2)
print(f"Largest gap over {len(check)} years: {check['gap'].abs().max():.2f}")
check.sort_index().tail(3)

## 4.10 When a site blocks scripts: download by hand

In [ ]:
manual = Path("data/manual/rbi_hbs_exchange_rate.xlsx")

if manual.exists():
    peek = pd.read_excel(manual, header=None, nrows=10)
    print(peek)        # look first, then choose skiprows= for the real read
else:
    print(f"{manual} not found.\n"
          "Download the exchange-rate table from the RBI Handbook of Statistics "
          "on the Indian Economy (www.rbi.org.in) in your browser, save it "
          "there, and note the URL and date in data/manual/SOURCES.txt.")

## 4.11 Terms of use and reproducibility

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading